In [1]:
!git clone https://github.com/index-tts/index-tts.git
%cd /content/index-tts
!pip install -q -U uv
!uv sync --all-extras

Cloning into 'index-tts'...
remote: Enumerating objects: 1990, done.
remote: Counting objects: 100% (339/339), done.
remote: Compressing objects: 100% (179/179), done.
remote: Total 1990 (delta 257), reused 160 (delta 160), pack-reused 1651 (from 3)
Receiving objects: 100% (1990/1990), 36.94 MiB | 32.33 MiB/s, done.
Resolving deltas: 100% (969/969), done.
/content/index-tts
Using CPython 3.11.13
Creating virtual environment at: .venv
Resolved 189 packages in 1ms
Prepared 177 packages in 2m 22s
Installed 177 packages in 847ms
 + absl-py==2.3.1
 + accelerate==1.8.1
 + aiofiles==24.1.0
 + annotated-types==0.7.0
 + antlr4-python3-runtime==4.9.3
 + anyio==4.10.0
 + argbind==0.3.9
 + asttokens==3.0.0
 + audioread==3.0.1
 + brotli==1.1.0
 + certifi==2025.8.3
 + cffi==2.0.0
 + charset-normalizer==3.4.3
 + click==8.2.1
 + cn2an==0.5.22
 + contourpy==1.3.3
 + cycler==0.12.1
 + cython==3.0.7
 + decorator==5.2.1
 + deepspeed==0.17.1
 + descript-audiotools==0.7.2
 + distance==0.1.3
 + docstring-par

In [2]:
!uv tool install "huggingface-hub"
!uv run huggingface-cli download IndexTeam/IndexTTS-2.5 --local-dir=checkpoints

Resolved 15 packages in 366ms
Prepared 10 packages in 255ms
Installed 15 packages in 25ms
 + anyio==4.16.0
 + click==8.5.0
 + filelock==4.2.9
 + fsspec==2026.9.0
 + h11==0.16.0
 + hf-xet==1.8.0
 + httpcore2==2.16.0
 + httpx2==2.16.0
 + huggingface-hub==2.4.0
 + idna==3.20
 + packaging==26.3
 + pyyaml==6.0.3
 + tqdm==4.70.1
 + truststore==0.10.5
 + typing-extensions==4.16.0
Installed 2 executables: hf, tiny-agents
⚠️  Warning: 'huggingface-cli download' is deprecated. Use 'hf download' instead.
Fetching 22 files:   0% 0/22 [00:00<?, ?it/s]Downloading 'codec.pth' to 'checkpoints/.cache/huggingface/download/Gl94AueGvIM0KbP8ia-vx4-hMMs=.d15cbed16a40f478438c961fb043f68dfa6353bf56c966761315db3433e9722c.incomplete'

LICENSE: 10.6kB [00:00, 7.59MB/s]
Download complete. Moving file to checkpoints/LICENSE

.gitattributes: 1.59kB [00:00, 7.07MB/s]
Download complete. Moving file to checkpoints/.gitattributes

README.md: 4.27kB [00:00, 11.7MB/s]
Download complete. Moving file to checkpoints/README.

In [3]:
!uv pip install fastapi uvicorn python-multipart

Using Python 3.13.16 environment at: /usr
Checked 3 packages in 142ms


In [4]:
%%writefile /content/index-tts/tts_server.py
import os, re, subprocess, threading, uuid
from typing import Optional, Dict, Any
import uvicorn
from fastapi import FastAPI, Depends, HTTPException, UploadFile, File, Form, Header
from fastapi.responses import Response
from pydantic import BaseModel
from indextts.infer_v2_5 import IndexTTS2

KEY = os.environ.get("ULTRON_KEY", "")
VOICES, OUT = "voices", "out"
os.makedirs(VOICES, exist_ok=True)
os.makedirs(OUT, exist_ok=True)

# model loads ONCE here, stays in VRAM
tts = IndexTTS2(cfg_path="checkpoints/config.yaml", model_dir="checkpoints", use_bf16=True)
lock = threading.Lock()
LANG = {"en": "EN", "zh": "ZH", "ja": "JP", "jp": "JP", "es": "ES", "ar": "AR"}
app = FastAPI()


def auth(authorization: str = Header(default="")):
    if KEY and authorization != f"Bearer {KEY}":
        raise HTTPException(401, "bad key")


def vpath(name):
    return os.path.join(VOICES, re.sub(r"[^A-Za-z0-9_-]", "", name) + ".wav")


class Speech(BaseModel):
    model: Optional[str] = None
    input: str
    voice: str
    response_format: str = "wav"
    speed: float = 1.0
    extra_params: Dict[str, Any] = {}


@app.get("/health")
def health():
    return {"ok": True}


@app.post("/v1/audio/voices")
def add_voice(audio_sample: UploadFile = File(...), name: str = Form(...),
              consent: str = Form(...), speaker_description: str = Form(""),
              _=Depends(auth)):
    raw = os.path.join(VOICES, "_" + uuid.uuid4().hex)
    with open(raw, "wb") as f:
        f.write(audio_sample.file.read())
    r = subprocess.run(["ffmpeg", "-y", "-i", raw, "-ac", "1", "-ar", "22050", "-t", "15", vpath(name)],
                       capture_output=True)
    os.remove(raw)
    if r.returncode:
        raise HTTPException(400, r.stderr.decode()[-300:])
    return {"voice": name}


@app.get("/v1/audio/voices")
def list_voices(_=Depends(auth)):
    return {"voices": [f[:-4] for f in os.listdir(VOICES) if f.endswith(".wav")]}


# @app.post("/v1/audio/speech")
# def speech(req: Speech, _=Depends(auth)):
#     p = vpath(req.voice)
#     if not os.path.exists(p):
#         raise HTTPException(404, f"voice '{req.voice}' not found. POST /v1/audio/voices first")
#     lang = LANG.get(str(req.extra_params.get("lang", "en")).lower(), "EN")
#     out = os.path.join(OUT, uuid.uuid4().hex + ".wav")
#     kw = {}
#     if req.speed and req.speed != 1.0:
#         kw["duration_factor"] = 1.0 / req.speed   # >1.0 duration = slower
#     with lock:
#         tts.infer(spk_audio_prompt=p, text=req.input, lang=lang,
#                   output_path=out, verbose=False, **kw)
#     if not os.path.exists(out):
#         raise HTTPException(500, "generation failed, check tts.log")
#     data = open(out, "rb").read()
#     os.remove(out)
#     return Response(data, media_type="audio/wav")

@app.post("/v1/audio/speech")
def speech(req: Speech, _=Depends(auth)):
    p = vpath(req.voice)
    if not os.path.exists(p):
        raise HTTPException(404, f"voice '{req.voice}' not found. POST /v1/audio/voices first")
    ep = req.extra_params
    lang = LANG.get(str(ep.get("lang", "en")).lower(), "EN")
    out = os.path.join(OUT, uuid.uuid4().hex + ".wav")
    kw = {}
    if req.speed and req.speed != 1.0:
        kw["duration_factor"] = 1.0 / req.speed   # >1.0 duration = slower
    vec = ep.get("emo_vector")
    if vec is not None:
        if len(vec) != 8:
            raise HTTPException(400, "emo_vector needs 8 floats")
        kw["emo_vector"] = [float(x) for x in vec]
        kw["use_random"] = False
    elif ep.get("emo_audio"):
        e = vpath(ep["emo_audio"])
        if not os.path.exists(e):
            raise HTTPException(404, f"emo_audio voice '{ep['emo_audio']}' not uploaded")
        kw["emo_audio_prompt"] = e
    if "emo_alpha" in ep:
        kw["emo_alpha"] = float(ep["emo_alpha"])
    kw["num_beams"] = int(req.extra_params.get("num_beams", 1))
    kw["interval_silence"] = 100
    with lock:
        tts.infer(spk_audio_prompt=p, text=req.input, lang=lang,
                  output_path=out, verbose=False, **kw)
    if not os.path.exists(out):
        raise HTTPException(500, "generation failed, check tts.log")
    data = open(out, "rb").read()
    os.remove(out)
    return Response(data, media_type="audio/wav")


if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8092)

Writing /content/index-tts/tts_server.py


In [5]:
import os, subprocess, time, urllib.request, secrets
os.chdir("/content/index-tts")
KEY = secrets.token_urlsafe(20)
subprocess.Popen(["uv", "run", "--no-sync", "python", "tts_server.py"],
                 env={**os.environ, "ULTRON_KEY": KEY},
                 stdout=open("tts.log", "w"), stderr=subprocess.STDOUT)
for i in range(120):
    time.sleep(5)
    try:
        urllib.request.urlopen("http://localhost:8092/health"); print("TTS server online"); break
    except Exception:
        if i % 6 == 0:
            print("loading... log tail:\n", open("tts.log").read()[-300:])
else:
    print(open("tts.log").read()[-2000:])
print("ULTRON_KEY =", KEY)

loading... log tail:
 
loading... log tail:
 
loading... log tail:
 
loading... log tail:
 lation. 
W1106 05:44:43.358000 2319 .venv/lib/python3.11/site-packages/torch/utils/cpp_extension.py:2425] If this is not desired, please set os.environ['TORCH_CUDA_ARCH_LIST'] to specific architectures.
>> QwenEmotion not loaded (use_qwen_emo=False)
>> GPT weights restored from: checkpoints/gpt.pth

loading... log tail:
 orch/csrc/api/include -isystem /usr/local/cuda/include -isystem /root/.local/share/uv/python/cpython-3.11.13-linux-x86_64-gnu/include/python3.11 -fPIC -std=c++17 -O3 -c /content/index-tts/indextts/s2mel/modules/bigvgan/alias_free_activation/cuda/anti_alias_activation.cpp -o anti_alias_activation.o 

loading... log tail:
 orch/csrc/api/include -isystem /usr/local/cuda/include -isystem /root/.local/share/uv/python/cpython-3.11.13-linux-x86_64-gnu/include/python3.11 -fPIC -std=c++17 -O3 -c /content/index-tts/indextts/s2mel/modules/bigvgan/alias_free_activation/cuda/anti_alias_activa

In [6]:
import re, subprocess, time
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8092"],
                 stderr=open("cf.log", "w"))
for _ in range(30):
    time.sleep(1)
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", open("cf.log").read())
    if m:
        print("ULTRON_TTS_URL =", m.group(0)); print("ULTRON_KEY     =", KEY); break

ULTRON_TTS_URL = https://buying-venues-dicke-singer.trycloudflare.com
ULTRON_KEY     = djVJdLZO9uHGxRsKtWL4L4Rru_A


In [7]:
# !pkill -f tts_server